# CrowdFlow 3D worker — Colab runtime

Runs the `crowdflow-colab-3d-worker` service on a free Colab GPU (T4), then
exposes it through a public tunnel. Point CrowdFlow's backend at it:

```
TWIN_PROVIDER=colab
TWIN_COLAB_URL=https://<your-tunnel>.ngrok-free.app
TWIN_COLAB_MODEL=hunyuan3d
```

The default model is **Hunyuan3D-2mini** (`hy3dgen`, Tencent) — it pip-installs
cleanly on the free T4 (~6 GB VRAM for shape generation). TRELLIS is NOT used:
its pinned `torch==2.4.0+cu121` wheel chain was removed from the PyTorch index
in 2026 (`nvidia-cudnn-cu12==9.1.0.70` no longer resolves), so Kaolin's `_C.so`
crashes with an ABI mismatch (`undefined symbol: _ZNK3c105Error4whatEv`).

The worker reports honest provenance: if the requested AI model is not
available it falls back to the deterministic generator and labels the result
`PROCEDURAL`.

## 1. Mount the worker source

The service lives in the CrowdFlow repo at `infrastructure/colab_3d_worker`.
Upload it to this notebook (Files panel → upload `colab_3d_worker/`), or
clone the repo and set the path below.

In [1]:
## 1. Mount the worker source

import os, sys
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_ROOT = "/content/drive/MyDrive"
WORKER_FOLDER_NAME = "colab_3d_worker"   # Change if your folder name differs
WORKER_PATH = os.path.join(DRIVE_ROOT, WORKER_FOLDER_NAME)

if not os.path.isdir(WORKER_PATH):
    raise SystemExit(f"Worker source not found at {WORKER_PATH}. Please upload your '{WORKER_FOLDER_NAME}' folder to '{DRIVE_ROOT}'.")

# Set CONTENT_DIR to the worker's parent folder (so the module can be imported)
CONTENT_DIR = DRIVE_ROOT

# Add to sys.path so we can import "colab_3d_worker"
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f"Worker path: {WORKER_PATH}")
print(f"Content directory (cwd for worker): {CONTENT_DIR}")

Mounting Google Drive...
Mounted at /content/drive
Worker path: /content/drive/MyDrive/colab_3d_worker
Content directory (cwd for worker): /content/drive/MyDrive


## 2. Install dependencies

The default AI model is **Hunyuan3D-2mini** via the `hy3dgen` pip package
(Tencent Hunyuan3D-2). Shape generation needs ~6 GB VRAM and runs on the free
T4. Texture generation needs more memory, so this worker generates the bare
3D shape only (GLB), which is exactly what the CrowdFlow twin renderer needs.

If `hy3dgen` is missing the worker still runs: it falls back to the
deterministic `PROCEDURAL` generator and labels the result honestly.

In [2]:
# --- Install Hunyuan3D-2mini (hy3dgen) on the kernel's Python ---
# NOTE: use the SAME python the worker will run under (sys.executable), so the
# uvicorn process can import hy3dgen. Do NOT pin an old torch: Colab ships a
# recent CUDA build already (torch>=2.6 required by transformers' torch.load
# safety check).

import sys
print(f"Python: {sys.version}")
print(f"sys.executable: {sys.executable}")

!{sys.executable} -m pip install -q --upgrade pip
print("--- Installing hy3dgen (Hunyuan3D-2, shape gen pipeline) ---")
!{sys.executable} -m pip install -q hy3dgen

print("--- Verifying hy3dgen imports ---")
!{sys.executable} -c "import hy3dgen; from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline; import torch; print('hy3dgen OK, torch', torch.__version__, 'cuda', torch.cuda.is_available())"

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
sys.executable: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.6 MB/s eta 0:00:00
--- Installing hy3dgen (Hunyuan3D-2, shape gen pipeline) ---
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.67.0 which is incompatible.
--- Verifying hy3dgen imports ---
hy3dgen OK, torch 2.11.0+cpu cuda False


## 3. (Optional) API-key-only models

Tripo / Meshy are hosted REST APIs and need no local GPU, only a key:

In [3]:
## 3. (Optional) API-key-only models

# Set your keys if you use Tripo / Meshy
# os.environ.setdefault("TRIPO_API_KEY", "your_key_here")
# os.environ.setdefault("MESHY_API_KEY", "your_key_here")

# Default model requested by the backend (hunyuan3d, trellis, meshy, tripo, ...)
os.environ.setdefault("COLAB_MODEL", "hunyuan3d")

# Optional Hunyuan3D tuning (defaults target the free T4):
# os.environ.setdefault("HUNYUAN3D_MODEL", "tencent/Hunyuan3D-2mini")
# os.environ.setdefault("HUNYUAN3D_SUBFOLDER", "hunyuan3d-dit-v2-mini")
# os.environ.setdefault("HUNYUAN3D_STEPS", "30")
# os.environ.setdefault("HUNYUAN3D_OCTREE_RESOLUTION", "320")
# os.environ.setdefault("HUNYUAN3D_NUM_CHUNKS", "8000")

print("Model environment variables ready.")

Model environment variables ready.


## 4. Start the worker

The server runs on port 8097 inside this runtime, using the kernel's Python
so it sees the `hy3dgen` packages installed in cell 2.

In [10]:
## 4. Start the worker

import subprocess, time, sys, os

# Ensure CONTENT_DIR is defined
try:
    CONTENT_DIR
except NameError:
    CONTENT_DIR = "/content/drive/MyDrive"

# Kill any process currently using port 8097 to avoid 'Address already in use'
!fuser -k 8097/tcp || true
time.sleep(2)

# Run with the KERNEL's python so hy3dgen (installed in cell 2) is importable.
python_exe = sys.executable

# Start uvicorn with the worker app
proc = subprocess.Popen(
    [python_exe, "-m", "uvicorn", "colab_3d_worker.worker:app",
     "--host", "0.0.0.0", "--port", "8097"],
    cwd=CONTENT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Starting worker...")
time.sleep(15)

# Check if process is still alive
if proc.poll() is not None:
    print("Worker failed to start. Error logs:")
    print(proc.stdout.read())
else:
    try:
        import urllib.request
        response = urllib.request.urlopen("http://127.0.0.1:8097/health", timeout=15).read().decode()
        print("Health check response:", response)
        print(f"Worker PID: {proc.pid}")
    except Exception as e:
        print(f"Worker is running but health check failed: {e}")

Starting worker...
Health check response: {"status":"ok","service":"crowdflow-colab-3d-worker","version":"1.0.0","default_model":"hunyuan3d","models":{"trellis":{"display":"TRELLIS (Microsoft, GPU)","available":false,"reason":"trellis package not installed (ModuleNotFoundError)"},"hunyuan3d":{"display":"Hunyuan3D (Tencent, GPU)","available":true,"reason":null},"meshy":{"display":"Meshy (hosted API)","available":false,"reason":"MESHY_API_KEY not configured"},"tripo":{"display":"Tripo (hosted API)","available":false,"reason":"TRIPO_API_KEY not configured"},"procedural":{"display":"Deterministic fallback","available":true,"reason":null}},"active_jobs":0}
Worker PID: 5353


## 5. Expose the worker with a tunnel

Copy the printed public URL into `TWIN_COLAB_URL` on the CrowdFlow backend.
Two options: ngrok (token required) or Cloudflare `cloudflared` (no account).

In [5]:
# 1. Install pyngrok for the kernel's Python
import sys
!{sys.executable} -m pip install -q pyngrok --upgrade

# 2. Verify installation
!{sys.executable} -m pip show pyngrok | grep -E "^Version:" || true
!{sys.executable} -c "import pyngrok; print('pyngrok version:', pyngrok.__version__)"

Version: 8.1.2
pyngrok version: 8.1.2


In [11]:
import getpass
import os
from pyngrok import ngrok

# 1. Kill any existing background ngrok processes
os.system("pkill -f ngrok || true")

# 2. Get and set the ngrok auth token
# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual token string,
# or leave it as "" to prompt securely in the environment/notebook.
AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

if not AUTH_TOKEN or AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("Please grab your token from: https://dashboard.ngrok.com/get-started/your-authtoken")
    AUTH_TOKEN = getpass.getpass("Enter your ngrok auth token: ").strip()

try:
    # Set authtoken using pyngrok API
    ngrok.set_auth_token(AUTH_TOKEN)
    print("Auth token set successfully.")
except Exception as e:
    print(f"Failed to set auth token: {e}")

# 3. Open the web tunnel
PORT = 8097

try:
    print(f"Connecting to ngrok on port {PORT}...")
    tunnel = ngrok.connect(PORT, "http")

    print("\n--- Tunnel Active ---")
    print(f"TWIN_COLAB_URL={tunnel.public_url}")
    print("---------------------\n")
    print("Copy the URL above into your CrowdFlow backend settings.")

except Exception as e:
    print(f"\nngrok failed: {e}")
    print("\n--- Fallback Option (No Account Needed) ---")
    print("Run the next cell to use Cloudflare Tunnel instead.")

Please grab your token from: https://dashboard.ngrok.com/get-started/your-authtoken
Enter your ngrok auth token: ··········
Auth token set successfully.
Connecting to ngrok on port 8097...

--- Tunnel Active ---
TWIN_COLAB_URL=https://growing-acts-cufflink.ngrok-free.dev
---------------------

Copy the URL above into your CrowdFlow backend settings.


In [7]:
# Cloudflare Tunnel fallback (no account needed). Run this cell if ngrok failed.

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!cloudflared tunnel --url http://127.0.0.1:8097

2026-08-14T04:35:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-14T04:35:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-14T04:35:20Z INF +--------------------------------------------------------------------------------------------+
2026-08-14T04:35:20Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-14T04:35:20Z INF |  https://propose-quest-rpm-disturbed.trycloudflare.com

## 6. Keep the runtime alive

Colab disconnects idle runtimes. Keep the session busy while CrowdFlow jobs
are running (a tab with this notebook open is usually enough). The worker
state lives in `_data/` next to the source and survives restarts.